# Quiz 5: arrival comparison graph

Upload the two simulation downloads from WUI-Go (baseline and your own case) and get the time versus cumulative arrivals graph.

1. In WUI-Go, open **Results**, select the run, choose **Download simulation outputs**. Do this for both runs. Do not unzip.
2. Rename the two zips so one name contains `baseline` and the other contains `own`.
3. Run the cell below and choose both zip files in the upload box. The two `final_agents.csv` files work too.

The graph is saved as `arrival_comparison.png`. Open the Files panel on the left to download it for your PDF. You still need to submit the original `final_agents.csv` files on bCourses.


In [ ]:
# Run this cell. Pick your two files when the upload box appears:
# the two ZIP files you downloaded from WUI-Go (baseline and your own case),
# or the two final_agents.csv files. Upload the baseline first.

import io, zipfile
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()

def read_final_agents(name, data):
    """Return final_agents.csv as a DataFrame from a zip or a csv upload."""
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            members = [m for m in z.namelist() if m.endswith("final_agents.csv")]
            if not members:
                raise ValueError(f"{name}: no final_agents.csv inside this zip")
            with z.open(members[0]) as f:
                return pd.read_csv(f)
    return pd.read_csv(io.BytesIO(data))

runs = []
for name, data in uploaded.items():
    df = read_final_agents(name, data)
    if "agent_status" not in df.columns or "arrival_time" not in df.columns:
        raise ValueError(f"{name}: this is not the original final_agents.csv "
                         f"(columns found: {list(df.columns)[:6]} ...)")
    lower = name.lower()
    label = "Baseline" if "baseline" in lower else ("Own case" if "own" in lower else name)
    runs.append((label, df))

if len(runs) < 2:
    print("Upload two files: baseline and your own case.")

plt.figure(figsize=(6, 4))
for label, df in runs:
    arrived = df[df["agent_status"] == "arrive"]
    t = arrived["arrival_time"].sort_values()
    plt.step(t / 60, range(1, len(t) + 1), where="post", label=label)
    n_total, n_arr = len(df), len(arrived)
    last = t.max() / 60 if n_arr else float("nan")
    print(f"{label}: {n_arr} of {n_total} vehicles arrived; "
          f"{n_total - n_arr} did not; last arrival at {last:.1f} min")

plt.xlabel("Elapsed time (min)")
plt.ylabel("Cumulative vehicles arrived")
plt.xlim(left=0); plt.ylim(bottom=0)
plt.legend()
plt.tight_layout()
plt.savefig("arrival_comparison.png", dpi=200)
plt.show()
print("Saved arrival_comparison.png (left panel > Files) for your PDF.")
